In [1]:
import os 
import joblib
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import TargetEncoder, OrdinalEncoder, StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor 
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score

In [2]:
df = pd.read_csv("job_salary_prediction_dataset.csv")

In [3]:
df.drop('certifications', axis=1, inplace=True)
# we are dropping because it may not affecting the salary as we saw in the bar graph

In [4]:
def bulid_pipeline(num_att, ord_att, one_att, tar_att, ord_cat): 
    num_pipe = Pipeline([
        ("stand", StandardScaler())
    ])

    ord_pipe = Pipeline([
        ('ordinal', OrdinalEncoder(categories=ord_cat))
    ])
    one_pipe = Pipeline([
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    tar_pipe = Pipeline([
        ('Target', TargetEncoder(target_type="continuous"))
    ])

    
    full_pipe = ColumnTransformer([
        ("num", num_pipe, num_att), 
        ("ord", ord_pipe, ord_att),
        ("one", one_pipe, one_att), 
        ("tar", tar_pipe, tar_att)
    ])

    return full_pipe

In [5]:
MODEL_FILE = 'model.pkl'
PIPELINE_FILE = 'pipeline.pkl'

In [8]:
if not os.path.exists(MODEL_FILE): 

    y = df['salary']
    x = df.drop("salary", axis=1)

    x_train, x_test, y_train, y_test = train_test_split(x, y, random_state=42, test_size=0.2)

    x_test = x_test.join(y_test)
    x_test.to_csv("input.csv", index=False)
    
    num_att = ['experience_years', 'skills_count']
    ord_att = ['education_level', 'company_size', 'remote_work']
    one_att = ['industry']
    tar_att = ['job_title', 'location']

    ord_cat = [
        # education level
        ['High School', 'Diploma', 'Bachelor', 'Master', 'PhD'], 

        # Company size 
        ['Startup', 'Small', 'Medium', 'Large', 'Enterprise'],

        # remote work
        ['No', 'Hybrid', 'Yes']
    ]

    pipeline = bulid_pipeline(num_att, ord_att, one_att, tar_att, ord_cat)

    x_transformed = pipeline.fit_transform(x_train, y_train)

    model = RandomForestRegressor(random_state=42)
    model.fit(x_transformed, y_train)
    
    train_preds = model.predict(x_transformed)
    print(f"Train R²  : {r2_score(y_train, train_preds):.4f}")
    print(f"Train RMSE: {root_mean_squared_error(y_train, train_preds):,.0f}")

    joblib.dump(model, MODEL_FILE)
    joblib.dump(pipeline, PIPELINE_FILE)

    print("Succesfully saved and trained the data !")

else:
    model = joblib.load(MODEL_FILE)
    pipeline = joblib.load(PIPELINE_FILE) 

    input_data = pd.read_csv("input.csv")

    y_actual = input_data['salary']
    x_input  = input_data.drop('salary', axis=1)

    x_transformed = pipeline.transform(x_input)
    model_pre = model.predict(x_transformed)
 
    input_data['prediction'] = model_pre
    input_data.to_csv("output.csv", index=False)

    r2   = r2_score(y_actual, model_pre)
    rmse = root_mean_squared_error(y_actual, model_pre)

    print(f"Test R2 square  : {r2:.4f}")
    print(f"Test RMSE: {rmse:,.0f}")
    print(rcv)

    print("Successfully predicted salary check out output.csv file!")
    

Train R²  : 0.9956
Train RMSE: 2,491
Succesfully saved and trained the data !
